In [ ]:
# 1. 安裝永豐 API 套件 (Colab 每次重開都要裝一次)
!pip install shioaji -q

import shioaji as sj
import pandas as pd
import numpy as np
import os
import time
from datetime import datetime, timedelta
from google.colab import drive
from tqdm.auto import tqdm

print("🚀 啟動台股微觀歷史挖掘機...")

# 2. 掛載雲端硬碟
drive.mount('/content/drive')

# ==========================================
# ⚙️ 設定區 (請修改成你自己的路徑與憑證)
# ==========================================
# 永豐 API 憑證
API_KEY = "J9udQP7KnGkKDdszyJ8nx8zmsmDahTtHRi8NzugPJ4pe"
SECRET_KEY = "9E8LBVqUYP2zVZiwHGYw3VWMsX325oxaoQij6dpM9Z8S"

# 資料庫路徑
TRACKER_PATH = '/content/drive/MyDrive/MarketMamba_Database/shioaji_tracker.csv'
SAVE_DIR = '/content/drive/MyDrive/MarketMamba_Database/Micro_5min'

# 挖掘參數
CHUNK_DAYS = 30   # 每次每檔股票往前挖 30 天
BATCH_SIZE = 100   # 每天幫 50 檔股票挖歷史 (視 API 穩定度可微調)
TARGET_DATE = '2016-01-01' # 歷史資料目標年份

# ==========================================
# 🔌 登入 API
# ==========================================
api = sj.Shioaji()
try:
    api.login(api_key=API_KEY, secret_key=SECRET_KEY)
    print("✅ 永豐 API 登入成功！")
except Exception as e:
    print(f"❌ 登入失敗，請檢查憑證。錯誤訊息: {e}")
    # 登入失敗就提早結束
    import sys
    sys.exit()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 786.5/786.5 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 31.8 MB/s eta 0:00:00


2026-03-05 09:54:28.990 | WARNING  | importlib._bootstrap:_call_with_frames_removed:488 - Optional: pip install shioaji[speed] or uv add shioaji --extra speed for better performance.


🚀 啟動台股微觀歷史挖掘機...
Mounted at /content/drive
Response Code: 0 | Event Code: 0 | Info: host '210.59.255.161:80', hostname '210.59.255.161:80' IP 210.59.255.161:80 (host 1 of 1) (host connection attempt 1 of 1) (total connection attempt 1 of 1) | Event: Session up
✅ 永豐 API 登入成功！


In [ ]:
!pip install shioaji -q
import shioaji as sj
import pandas as pd

print("🩺 永豐 API 存活測試...")
api = sj.Shioaji()
api.login(api_key="J9udQP7KnGkKDdszyJ8nx8zmsmDahTtHRi8NzugPJ4pe", secret_key="9E8LBVqUYP2zVZiwHGYw3VWMsX325oxaoQij6dpM9Z8S")

# 只抓台積電昨天的資料
contract = api.Contracts.Stocks['2330']
kbars = api.kbars(contract, start='2026-03-04', end='2026-03-04')
df = pd.DataFrame({**kbars})

if df.empty:
    print("🚨 確診！台積電昨天的資料都抓不到，永豐 API 目前處於『額度耗盡 / 深夜宵禁』狀態！")
else:
    print(f"✅ 破案！API 活著 (抓到 {len(df)} 筆)。這代表是我們剛剛的程式邏輯有鬼！")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 786.5/786.5 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 42.2 MB/s eta 0:00:00


2026-03-06 12:24:20.864 | WARNING  | importlib._bootstrap:_call_with_frames_removed:488 - Optional: pip install shioaji[speed] or uv add shioaji --extra speed for better performance.


🩺 永豐 API 存活測試...
Response Code: 0 | Event Code: 0 | Info: host '210.59.255.161:80', hostname '210.59.255.161:80' IP 210.59.255.161:80 (host 1 of 1) (host connection attempt 1 of 1) (total connection attempt 1 of 1) | Event: Session up
Response Code: 200 | Event Code: 16 | Info: APISUB/V1/SYS/CONTRACT | Event: Subscribe or Unsubscribe ok
🚨 確診！台積電昨天的資料都抓不到，永豐 API 目前處於『額度耗盡 / 深夜宵禁』狀態！


In [ ]:
import os
import pandas as pd
from tqdm.auto import tqdm
from google.colab import drive
drive.mount('/content/drive')
print("🩺 啟動 1 分鐘線資料庫 X 光掃描機...\n")

# 📂 這裡請替換成你實際存放 1 分鐘線 Parquet 檔的資料夾路徑
# 假設是這個路徑，如果不同請幫我微調一下
DATA_DIR = '/content/drive/MyDrive/MarketMamba_Database/Micro_5min_History'

results = []

# 抓出所有 parquet 檔案
if not os.path.exists(DATA_DIR):
    print(f"❌ 找不到資料夾: {DATA_DIR}，請確認路徑！")
else:
    files = [f for f in os.listdir(DATA_DIR) if f.endswith('.parquet')]
    print(f"📂 發現 {len(files)} 個股票檔案，準備開始體檢...")

    for f in tqdm(files, desc="資料健康度掃描中"):
        ticker = f.split('_')[0]
        path = os.path.join(DATA_DIR, f)

        try:
            df = pd.read_parquet(path)

            # 檢查是否為空檔案
            if df.empty:
                results.append({'Ticker': ticker, 'Status': 'Empty (空殼)', 'Rows': 0, 'Oldest': None, 'Newest': None})
                continue

            # 抓取第一筆(最老)和最後一筆(最新)的時間
            # 假設你的時間欄位是 index，如果不是的話，請把 df.index 改成 df['Date'] 或你的時間欄位名稱
            oldest = df.index.min()
            newest = df.index.max()
            rows = len(df)

            results.append({
                'Ticker': ticker,
                'Status': 'Healthy (健康)',
                'Rows': rows,
                'Oldest': oldest,
                'Newest': newest
            })

        except Exception as e:
            results.append({'Ticker': ticker, 'Status': f'Error ({str(e)})', 'Rows': 0, 'Oldest': None, 'Newest': None})

    # 轉換成 DataFrame 方便統計
    summary_df = pd.DataFrame(results)

    # ==========================================
    # 📊 產出健康檢查報告
    # ==========================================
    healthy_df = summary_df[summary_df['Status'] == 'Healthy (健康)'].copy()

    print("\n==========================================")
    print("📋 【1 分鐘線資料庫健康檢查報告】")
    print("==========================================")
    print(f"📁 總掃描檔案數: {len(summary_df)}")
    print(f"✅ 健康檔案數:   {len(healthy_df)}")
    print(f"⚠️ 異常檔案數:   {len(summary_df) - len(healthy_df)}")
    print("------------------------------------------")

    if not healthy_df.empty:
        # 計算每檔股票平均擁有的 1 分K 數量
        avg_rows = healthy_df['Rows'].mean()
        print(f"📈 平均每檔股票擁有: {avg_rows:,.0f} 根 1 分 K 線")

        # 分析「歷史岩床」的年份分佈
        healthy_df['Oldest_Year'] = pd.to_datetime(healthy_df['Oldest']).dt.year
        print("\n📅 【歷史岩床年份分佈 (最老資料落在哪一年)】:")
        year_counts = healthy_df['Oldest_Year'].value_counts().sort_index()
        for year, count in year_counts.items():
            print(f"   {int(year)} 年: {count} 檔股票")

        print("\n🔍 【隨機抽查 5 檔股票詳細資料】:")
        print(healthy_df[['Ticker', 'Rows', 'Oldest', 'Newest']].sample(min(5, len(healthy_df))).to_string(index=False))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🩺 啟動 1 分鐘線資料庫 X 光掃描機...

📂 發現 1946 個股票檔案，準備開始體檢...


資料健康度掃描中:   0%|          | 0/1946 [00:00<?, ?it/s]


📋 【1 分鐘線資料庫健康檢查報告】
📁 總掃描檔案數: 1946
✅ 健康檔案數:   597
⚠️ 異常檔案數:   1349
------------------------------------------
📈 平均每檔股票擁有: 31,031 根 1 分 K 線

📅 【歷史岩床年份分佈 (最老資料落在哪一年)】:
   2024 年: 278 檔股票
   2025 年: 319 檔股票

🔍 【隨機抽查 5 檔股票詳細資料】:
Ticker  Rows              Oldest              Newest
  1439  2969 2025-01-02 09:21:00 2026-03-06 13:30:00
  2067  2799 2024-07-30 09:34:00 2025-09-03 12:39:00
  3010 41237 2025-01-02 09:01:00 2026-03-06 13:30:00
  2338 42445 2024-07-30 09:01:00 2025-09-03 13:30:00
  1435  1036 2025-01-02 10:42:00 2026-03-06 13:25:00


In [ ]:
!pip install shioaji -q
import os
import time
import datetime
import pandas as pd
import shioaji as sj
from tqdm.auto import tqdm
from google.colab import drive
from tqdm.auto import tqdm
drive.mount('/content/drive')

print("⛏️ 啟動 1 分鐘線每日游擊採礦機 (自動續傳版)...\n")

# ==========================================
# 📂 1. 路徑與參數設定
# ==========================================
UNIVERSE_PATH = '/content/drive/MyDrive/MarketMamba_Database/final_universe.csv'
DATA_DIR = '/content/drive/MyDrive/MarketMamba_Database/Micro_5min_History'

# 確保資料夾存在
os.makedirs(DATA_DIR, exist_ok=True)

# 設定抓取的時間區間 (永豐 1 分 K 通常極限是 1 年多，我們設寬一點)
# 這裡預設抓取 2025-01-01 到今天
START_DATE = "2025-01-01"
END_DATE = datetime.datetime.now().strftime("%Y-%m-%d")

# ==========================================
# 📊 2. 盤點進度：嚴格審查已存在檔案的健康度 (裝甲升級版)
# ==========================================
import os
import pandas as pd

# 讀取全市場名單
universe_df = pd.read_csv(UNIVERSE_PATH, dtype=str)
universe_df['Ticker'] = universe_df['Ticker'].astype(str).str.zfill(4)
all_tickers = set(universe_df['Ticker'].tolist())

done_tickers = set()
existing_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.parquet')]

print("🩺 正在嚴格檢查硬碟內既有檔案的健康度...")

for f in existing_files:
    ticker = f.split('_')[0]
    file_path = os.path.join(DATA_DIR, f)

    try:
        # 嘗試打開檔案
        df = pd.read_parquet(file_path)

        # 情況 A：檔案有資料 (健康) -> 標記完工
        if not df.empty:
            done_tickers.add(ticker)

        # 情況 B：檔案是空的，且是我們特別標記的殭屍股 -> 標記完工 (不浪費流量去抓)
        elif df.empty:
            done_tickers.add(ticker)

    except Exception as e:
        # 情況 C：檔案壞掉、讀不出來 (破圖) -> 刪除它！把它踢出完工名單，稍後重抓！
        print(f"🗑️ 發現損壞檔案 {f}，已自動刪除準備重抓！(錯誤: {e})")
        os.remove(file_path)

remaining_tickers = sorted(list(all_tickers - done_tickers))

print(f"📋 全市場總目標: {len(all_tickers)} 檔")
print(f"✅ 嚴格檢查通過 (健康/已確認空股): {len(done_tickers)} 檔")
print(f"🎯 今日待命開挖: {len(remaining_tickers)} 檔\n")

if len(remaining_tickers) == 0:
    print("🎉 恭喜！全市場 1 分鐘線已經全部收集完畢！不需再執行。")
else:
    # ==========================================
    # 🚀 3. 登入 API 並開始推進
    # ==========================================
    print("🔌 正在連線永豐 Shioaji API...")
    api = sj.Shioaji(simulation=False)
    # 請替換成你的真實金鑰
    api.login("J9udQP7KnGkKDdszyJ8nx8zmsmDahTtHRi8NzugPJ4pe", "9E8LBVqUYP2zVZiwHGYw3VWMsX325oxaoQij6dpM9Z8S")
    print("✅ 登入成功！開挖！\n")

    progress_bar = tqdm(remaining_tickers, desc="今日採礦進度")

    for ticker in progress_bar:
        try:
            # 1. 取得合約
            contract = api.Contracts.Stocks.get(ticker)
            if not contract:
                # 找不到合約，存入空檔案作標記
                empty_df = pd.DataFrame()
                empty_df.to_parquet(os.path.join(DATA_DIR, f"{ticker}_1m.parquet"))
                continue

            # 2. 請求 1 分 K 線
            kbars = api.kbars(
                contract=contract,
                start=START_DATE,
                end=END_DATE
            )

            # 3. 轉成 DataFrame
            df = pd.DataFrame({**kbars})

            # 4. 處理時間戳記並設為 Index
            if not df.empty:
                df['ts'] = pd.to_datetime(df['ts'])
                df.set_index('ts', inplace=True)

            # 5. 存檔 (就算 df.empty 是空的，也會存一個空 parquet，防止明天又抓一次)
            save_path = os.path.join(DATA_DIR, f"{ticker}_1m.parquet")
            df.to_parquet(save_path)

            # 稍微停頓 0.2 秒，避免對 API 請求過於頻繁被 ban
            time.sleep(0.2)

        except Exception as e:
            error_msg = str(e).lower()
            # 🚨 捕捉到流量天花板！
            if "limit" in error_msg or "quota" in error_msg or "over" in error_msg:
                print(f"\n🛑 [系統煞車] 觸及永豐每日 API 流量天花板！")
                print(f"🛑 腳本已自動安全停止。今天辛苦了，我們明天繼續！")
                break # 直接跳出迴圈，結束今天的程式
            elif "timeout" in error_msg:
                print(f"\n⚠️ [網路超時] {ticker} 伺服器無回應，將跳過此檔 (明天會再試一次)。")
                time.sleep(2) # 遇到 Timeout 休息久一點
            else:
                print(f"\n⚠️ [未知錯誤] {ticker} 發生異常: {e}，跳過。")

    print("\n==========================================")
    print("🌙 今日游擊採礦任務結束！進度已自動保存。")
    print("==========================================")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⛏️ 啟動 1 分鐘線每日游擊採礦機 (自動續傳版)...

🩺 正在嚴格檢查硬碟內既有檔案的健康度...
📋 全市場總目標: 1937 檔
✅ 嚴格檢查通過 (健康/已確認空股): 1946 檔
🎯 今日待命開挖: 0 檔

🎉 恭喜！全市場 1 分鐘線已經全部收集完畢！不需再執行。


In [1]:
# ==========================================
# 🚀 執行前請確保已安裝 Shioaji 套件：
!pip install shioaji
# ==========================================
import os
import time
import datetime
import pandas as pd
from tqdm.auto import tqdm
import shioaji as sj
from google.colab import drive
drive.mount('/content/drive')


print("⛏️ 啟動 1 分鐘線終極採礦機 V2 (Shioaji 永豐自動修復版)...\n")

# ==========================================
# 🔑 1. 永豐 API 登入設定 (請填入你的 Key)
# ==========================================
YOUR_API_KEY = "J9udQP7KnGkKDdszyJ8nx8zmsmDahTtHRi8NzugPJ4pe"
YOUR_SECRET_KEY = "9E8LBVqUYP2zVZiwHGYw3VWMsX325oxaoQij6dpM9Z8S"

print("🔌 正在連線永豐 Shioaji 伺服器...")
api = sj.Shioaji()
try:
    api.login(
        api_key=YOUR_API_KEY,
        secret_key=YOUR_SECRET_KEY
    )
    print("✅ 永豐 API 登入成功！")
except Exception as e:
    print(f"❌ 登入失敗，請檢查 Key 是否正確。錯誤訊息: {e}")
    raise e

# ==========================================
# 📁 2. 基礎設定與名單對齊
# ==========================================
MACRO_DIR = '/content/drive/MyDrive/MarketMamba_Database/Macro_1D_History'
MICRO_DIR = '/content/drive/MyDrive/MarketMamba_Database/Micro_5min_History'
os.makedirs(MICRO_DIR, exist_ok=True)

target_tickers = [f.split('_')[0] for f in os.listdir(MACRO_DIR) if f.endswith('.parquet') and f != 'TAIEX_Macro.parquet']
print(f"🎯 鎖定全市場目標: {len(target_tickers)} 檔股票")

TODAY_STR = datetime.date.today().strftime("%Y-%m-%d")
DEFAULT_START = "2024-01-01"
task_queue = []

# ==========================================
# 🩺 3. 嚴格的 X 光體檢與任務分發
# ==========================================
print("\n🩺 正在進行全資料庫 X 光掃描與任務分發...")
for ticker in tqdm(target_tickers, desc="掃描中"):
    file_path = os.path.join(MICRO_DIR, f"{ticker}_1min.parquet")

    if not os.path.exists(file_path):
        task_queue.append({'ticker': ticker, 'mode': 'FULL', 'start': DEFAULT_START, 'path': file_path})
        continue

    try:
        df = pd.read_parquet(file_path)
        if df.empty:
            os.remove(file_path)
            task_queue.append({'ticker': ticker, 'mode': 'FULL', 'start': DEFAULT_START, 'path': file_path})
            continue

        newest_date = df.index.max()
        newest_str = newest_date.strftime("%Y-%m-%d")

        # 容許 3 天的假日落差，超過 3 天就啟動斷點續傳
        if newest_str < (datetime.date.today() - datetime.timedelta(days=3)).strftime("%Y-%m-%d"):
            resume_start = (newest_date + datetime.timedelta(days=1)).strftime("%Y-%m-%d")
            task_queue.append({'ticker': ticker, 'mode': 'APPEND', 'start': resume_start, 'path': file_path, 'old_df': df})

    except Exception as e:
        os.remove(file_path)
        task_queue.append({'ticker': ticker, 'mode': 'FULL', 'start': DEFAULT_START, 'path': file_path})

print(f"\n📋 體檢完成！共有 {len(task_queue)} 檔股票需要修復或下載。")

# ==========================================
# 🚀 4. 啟動自我修復採礦引擎 (Shioaji 抓取)
# ==========================================
if len(task_queue) > 0:
    print(f"🚀 開始執行修復與開採任務 (目標日期: {TODAY_STR})...")
    # --- 迴圈外面先宣告一個計數器 ---
    consecutive_empty_count = 0
    for task in tqdm(task_queue, desc="⛏️ 採礦進度"):
        ticker = task['ticker']
        mode = task['mode']
        start_date = task['start']
        save_path = task['path']

        try:
            # 取得股票合約
            contract = api.Contracts.Stocks.get(ticker)
            if not contract:
                print(f"⚠️ 找不到 {ticker} 的永豐合約，跳過。")
                continue

            # 呼叫 API 抓取 1 分鐘 K 線
            kbars = api.kbars(contract, start=start_date, end=TODAY_STR)
            new_df = pd.DataFrame({**kbars})

            # ==========================================
            # 🚨 升級版：連續空包彈偵測雷達
            # ==========================================
            if new_df.empty:
                consecutive_empty_count += 1
                print(f" ⚠️ 警告：{ticker} 抓無資料 (連續空包彈: {consecutive_empty_count}/10)")

                # 如果連續 10 檔都沒資料，直接強制斷線退場！
                if consecutive_empty_count >= 10:
                    print("\n🛑 偵測到連續 10 檔無資料！極高機率已達 500MB 流量上限，啟動強制煞車，明天再戰。")
                    break
                continue
            else:
                # 只要成功抓到一檔有資料的，計數器就立刻歸零！
                consecutive_empty_count = 0

            # 將 Shioaji 吐出的 timestamp (ts) 轉成 DateTimeIndex
            new_df['ts'] = pd.to_datetime(new_df['ts'])
            new_df.set_index('ts', inplace=True)

            # ==========================================
            # 🧩 資料融合與存檔 (防呆機制)
            # ==========================================
            if mode == 'APPEND':
                old_df = task['old_df']
                combined_df = pd.concat([old_df, new_df])
                # 去除重複時間點並排序
                combined_df = combined_df[~combined_df.index.duplicated(keep='last')].sort_index()
            else:
                combined_df = new_df.sort_index()

            # 存回硬碟
            combined_df.to_parquet(save_path)

            # 禮貌性暫停，避免被 Shioaji 伺服器踢下線
            time.sleep(1.5)

        except Exception as e:
            err_msg = str(e).lower()
            if 'limit' in err_msg or 'quota' in err_msg or 'over' in err_msg:
                print(f"\n🛑 觸發永豐 API 流量限制！程式優雅暫停，明天再戰。")
                break
            else:
                print(f"\n⚠️ 抓取 {ticker} 時發生未知錯誤: {e}，跳過換下一檔。")
                continue
else:
    print("\n🎉 太完美了！你的 1 分鐘線資料庫目前 100% 健康且最新，無需採礦！")

# 登出 API，釋放連線資源
api.logout()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 786.5/786.5 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 50.5 MB/s eta 0:00:00


2026-03-09 05:35:06.934 | WARNING  | importlib._bootstrap:_call_with_frames_removed:488 - Optional: pip install shioaji[speed] or uv add shioaji --extra speed for better performance.


Mounted at /content/drive
⛏️ 啟動 1 分鐘線終極採礦機 V2 (Shioaji 永豐自動修復版)...

🔌 正在連線永豐 Shioaji 伺服器...
Response Code: 0 | Event Code: 0 | Info: host '210.59.255.161:80', hostname '210.59.255.161:80' IP 210.59.255.161:80 (host 1 of 1) (host connection attempt 1 of 1) (total connection attempt 1 of 1) | Event: Session up
✅ 永豐 API 登入成功！
Response Code: 200 | Event Code: 16 | Info: APISUB/V1/SYS/CONTRACT | Event: Subscribe or Unsubscribe ok
🎯 鎖定全市場目標: 1937 檔股票

🩺 正在進行全資料庫 X 光掃描與任務分發...


掃描中:   0%|          | 0/1937 [00:00<?, ?it/s]


📋 體檢完成！共有 1721 檔股票需要修復或下載。
🚀 開始執行修復與開採任務 (目標日期: 2026-03-09)...


⛏️ 採礦進度:   0%|          | 0/1721 [00:00<?, ?it/s]

 ⚠️ 警告：4568 抓無資料 (連續空包彈: 1/10)
 ⚠️ 警告：4584 抓無資料 (連續空包彈: 2/10)
 ⚠️ 警告：5013 抓無資料 (連續空包彈: 3/10)

🛑 觸發永豐 API 流量限制！程式優雅暫停，明天再戰。


True